# 🤖 Prototipo del Motor de Recomendación Híbrido

Este notebook implementa la lógica avanzada de matching para RentAI, resolviendo los puntos ciegos detectados:
1. **Bilateralidad**: Filtro estricto de Dealbreakers.
2. **Presupuesto Elástico**: Scoring por proximidad de rangos.
3. **Conducta Exponencial**: Penalización fuerte por diferencias extremas en limpieza/social.
4. **NLP Semántico**: Similitud de biografía usando TF-IDF.

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. LUBRICACIÓN Y CARGA DE DATOS ---

def parse_postgres_array(val):
    if not isinstance(val, str) or not val.startswith('{'): return []
    clean = val.strip('{}').replace('""', '"')
    return [i.strip('"') for i in clean.split(",")] if clean else []

def parse_json(val):
    try: return json.loads(val)
    except: return {}

df = pd.read_csv('perfiles/perfiles_simulados_2000.csv')

# Aplicar parsers
df['interests'] = df['interests'].apply(parse_postgres_array)
df['lifestyle_tags'] = df['lifestyle_tags'].apply(parse_postgres_array)
df['exclusion_rules'] = df['exclusion_rules'].apply(parse_json)
df['importance_weights'] = df['importance_weights'].apply(parse_json)

print(f"✅ Base de datos cargada: {len(df)} perfiles.")

## 🛠️ El Motor del Algoritmo
Definimos las funciones de cálculo para cada capa del modelo híbrido.

In [ ]:
def score_financiero(u1, u2):
    """
    Cálculo de intersección de rangos.
    Si no hay solapamiento, calculamos la distancia inversa.
    """
    start = max(u1['min_budget'], u2['min_budget'])
    end = min(u1['max_budget'], u2['max_budget'])
    
    # Si hay solapamiento
    if start <= end:
        return 1.0
    
    # Si no hay solapamiento, calcular qué tan lejos están (penalización suave)
    distancia = start - end
    return max(0, 1 - (distancia / 500000)) # Decae a 0 si están a más de 500k de distancia

def score_conductal(u1, u2):
    """
    Limpieza y Social: diferencia exponencial.
    Diferencia 0 -> 1.0
    Diferencia 9 -> ~0.15
    """
    diff_clean = abs(u1['cleanliness_level'] - u2['cleanliness_level'])
    diff_social = abs(u1['social_level'] - u2['social_level'])
    
    # Promediamos la penalización exponencial
    score = (np.exp(-0.15 * diff_clean) + np.exp(-0.15 * diff_social)) / 2
    return score

def similarity_jaccard(list1, list2):
    if not list1 or not list2: return 0
    s1, s2 = set(list1), set(list2)
    return len(s1 & s2) / len(s1 | s2)

def verificar_exclusion_bilateral(u1, u2):
    """
    FILTRO CRÍTICO: Si A excluye a B O B excluye a A -> Match 0
    """
    # Regla: Fumadores
    if u1['exclusion_rules'].get('no_smokers') and "Fumador" in u2['lifestyle_tags']: return False
    if u2['exclusion_rules'].get('no_smokers') and "Fumador" in u1['lifestyle_tags']: return False
    
    # Regla: Mascotas
    # (Asumiendo que si pets_accepted es False, no quiere a alguien con el tag 'Mascotas')
    if not u1['exclusion_rules'].get('pets_accepted') and "Mascotas" in u2['lifestyle_tags']: return False
    if not u2['exclusion_rules'].get('pets_accepted') and "Mascotas" in u1['lifestyle_tags']: return False
    
    return True

## 🧬 Capa Semántica (NLP)
Convertimos las biografías en vectores TF-IDF para comparar el "vibe" de los usuarios.

In [ ]:
tfidf = TfidfVectorizer(stop_words=['soy', 'busco', 'una', 'con', 'que', 'en'])
tfidf_matrix = tfidf.fit_transform(df['bio'].fillna(''))

def score_semantico(index_a, index_b):
    sim = cosine_similarity(tfidf_matrix[index_a], tfidf_matrix[index_b])
    return sim[0][0]

## 🏆 Ejecución del Matching
Esta función integra todo y calcula el ranking final.

In [ ]:
def obtener_recomendaciones(user_id, top_n=10):
    target_idx = df[df['id'] == user_id].index[0]
    u1 = df.iloc[target_idx]
    
    resultados = []
    
    for i, u2 in df.iterrows():
        if i == target_idx: continue
        
        # 0. Exclusion
        if not verificar_exclusion_bilateral(u1, u2): continue
        
        # 1. Financiero
        s_fin = score_financiero(u1, u2)
        
        # 2. Conductal
        s_cond = score_conductal(u1, u2)
        
        # 3. Afinidad (Tags + Intereses)
        s_tag = similarity_jaccard(u1['lifestyle_tags'], u2['lifestyle_tags'])
        s_int = similarity_jaccard(u1['interests'], u2['interests'])
        s_afin = (s_tag * 0.7) + (s_int * 0.3) # El estilo de vida pesa más que los hobbies
        
        # 4. Semántico
        s_sem = score_semantico(target_idx, i)
        
        # --- INTEGRACIÓN CON PESOS --- 
        w = u1['importance_weights']
        # Normalización de pesos
        total_w = sum(w.values())
        score_final = (
            (s_fin * w.get('budget', 0.25)) +
            (s_cond * w.get('personality', 0.25)) +
            (s_afin * w.get('lifestyle', 0.25)) +
            (s_sem * w.get('interests', 0.25))
        ) / total_w
        
        resultados.append({
            'id': u2['id'],
            'nombre': f"{u2['first_name']} {u2['last_name']}",
            'score': round(score_final * 100, 2),
            'presupuesto': u2['monthly_budget'],
            'bio': u2['bio']
        })
    
    # Ordenar y devolver
    ranking = pd.DataFrame(resultados).sort_values(by='score', ascending=False)
    return ranking.head(top_n)

# Probar con el primer usuario del dataset
pivot_user_id = df.iloc[0]['id']
print(f"Buscando matches para: {df.iloc[0]['first_name']} (Cleanliness: {df.iloc[0]['cleanliness_level']})")
obtener_recomendaciones(pivot_user_id)